# Posterior parameter recovery

Load a saved SBI posterior artifact, inspect training details and learning curves, then run one fresh parameter-recovery simulation.


## Imports


In [ ]:
import copy
import json
import math
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

project_root = Path.cwd()
if not (project_root / "tasks").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    import scienceplots  # noqa: F401
except ModuleNotFoundError:
    scienceplots = None
else:
    plt.style.use("science")

from network_creation.model_inversion import load_posterior_artifact
from tasks.policies import prepare_policy_run
from notebooks.utils.table_pretty_print import make_gt_tabs, default_style_gt_table


project_root

## Load posterior artifact


In [ ]:
artifact_root = project_root / "artifacts" / "model_inversion"
artifact_candidates = sorted(
    [
        path
        for path in artifact_root.glob("*")
        if path.is_dir()
        and (path / "metadata.json").is_file()
        and (path / "posterior.pkl").is_file()
    ],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not artifact_candidates:
    raise FileNotFoundError(
        f"No complete posterior artifacts found under {artifact_root}"
    )

artifact_options = [
    (str(path.relative_to(project_root)), str(path.relative_to(project_root)))
    for path in artifact_candidates
]
artifact_selector = widgets.Dropdown(
    options=artifact_options,
    value=artifact_options[0][1],
    description="Artifact:",
    layout=widgets.Layout(width="95%"),
)
display(artifact_selector)

In [ ]:
artifact_dir = project_root / artifact_selector.value
artifact = load_posterior_artifact(artifact_dir)
metadata = artifact.metadata
dataset_metadata = metadata["dataset"]

artifact_summary = {
    "artifact_dir": str(artifact_dir.relative_to(project_root)),
    "created_at": metadata.get("created_at"),
    "posterior_type": type(artifact.posterior).__name__,
    "sbi_method": metadata.get("sbi", {}).get("method"),
    "amortized": metadata.get("sbi", {}).get("amortized"),
    "n_simulations": dataset_metadata.get("n_simulations"),
    "x_representation": dataset_metadata.get("x_representation"),
    "x_event_shape": tuple(dataset_metadata.get("x_event_shape", ())),
}

display(make_gt_tabs(artifact_summary, gt_transform=None))

## Metadata


In [ ]:
prior_low = dataset_metadata.get("prior_low", [])
prior_high = dataset_metadata.get("prior_high", [])

theta_df = (
    pd.DataFrame(
        {
            "theta": artifact.theta_order,
            "prior_low": pd.Series(prior_low),
            "prior_high": pd.Series(prior_high),
        }
    )
    .rename_axis("index")
    .reset_index()
)

observation_df = (
    pd.Series(artifact.observation_order, name="observable")
    .rename_axis("index")
    .reset_index()
)

constant_theta = dataset_metadata.get("constant_theta", {})

display(
    make_gt_tabs(
        [
            {
                "tab": "Theta order",
                "title": "Posterior theta order",
                "value": theta_df,
            },
            {
                "tab": "Observations",
                "title": "Observation order",
                "value": observation_df,
            },
            {
                "tab": "Dropped constants",
                "title": "Dropped constant theta values",
                "value": (
                    pd.Series(constant_theta, name="value")
                    .rename_axis("theta")
                    .reset_index()
                    if constant_theta
                    else None
                ),
            },
        ],
        tab_key="tab",
        title_key="title",
        gt_transform=None,
    )
)

In [ ]:
display(
    make_gt_tabs(
        [
            {
                "tab": "Density estimator",
                "title": "Density estimator",
                "value": metadata.get("density_estimator", {}),
            },
            {
                "tab": "Embedding",
                "title": "Embedding",
                "value": metadata.get("embedding", {}),
            },
            {
                "tab": "Training",
                "title": "Training",
                "value": metadata.get("training", {}),
            },
        ],
        tab_key="tab",
        title_key="title",
        gt_transform=None,
    )
)

## Learning curve


In [ ]:
LEARNING_CURVE_TAGS = ("training_loss", "validation_loss")
LEARNING_DIAGNOSTIC_TAGS = ("epochs_trained", "best_validation_loss")
LEARNING_SCALAR_TAGS = LEARNING_CURVE_TAGS + LEARNING_DIAGNOSTIC_TAGS


def load_scalars(log_dir):
    accumulator = EventAccumulator(str(log_dir))
    accumulator.Reload()
    scalar_tags = set(accumulator.Tags().get("scalars", []))

    scalars = {}
    for tag in LEARNING_SCALAR_TAGS:
        if tag in scalar_tags:
            events = accumulator.Scalars(tag)
            scalars[tag] = {
                "step": np.asarray([event.step for event in events], dtype=np.int64),
                "value": np.asarray(
                    [event.value for event in events], dtype=np.float64
                ),
            }

    return scalars


selected_log_dir = artifact.tensorboard_log_dir
learning_scalars = load_scalars(selected_log_dir) if selected_log_dir.is_dir() else {}


if not learning_scalars:
    display(Markdown(f"No learning curve found under `{selected_log_dir}`."))
else:
    curve_tags = [
        tag
        for tag in LEARNING_CURVE_TAGS
        if tag in learning_scalars and len(learning_scalars[tag]["value"])
    ]

    if curve_tags:
        fig, ax = plt.subplots(figsize=(8, 4))
        for tag in curve_tags:
            curve = learning_scalars[tag]
            ax.plot(curve["step"], curve["value"], label=tag)

        ax.set_xlabel("epoch")
        ax.set_ylabel("loss")
        ax.set_title("SBI training curve")
        ax.legend()
        plt.grid(alpha=0.25)
        plt.show()

    summary_df = pd.DataFrame(
        [
            {
                "tag": tag,
                "n_points": len(learning_scalars[tag]["value"]),
                "first": learning_scalars[tag]["value"][0],
                "last": learning_scalars[tag]["value"][-1],
                "min": np.min(learning_scalars[tag]["value"]),
                "max": np.max(learning_scalars[tag]["value"]),
            }
            for tag in curve_tags
        ]
    )
    diagnostics_df = pd.DataFrame(
        [
            {
                "tag": tag,
                "n_points": len(curve["value"]),
                "step": int(curve["step"][-1]),
                "value": float(curve["value"][-1]),
            }
            for tag in LEARNING_DIAGNOSTIC_TAGS
            if tag in learning_scalars
            for curve in [learning_scalars[tag]]
            if len(curve["value"])
        ]
    )

    display(
        make_gt_tabs(
            [
                {
                    "tab": "Curves",
                    "title": "Learning curve summary",
                    "value": summary_df,
                },
                {
                    "tab": "Diagnostics",
                    "title": "Training diagnostics",
                    "value": diagnostics_df,
                }
            ],
            tab_key="tab",
            title_key="title",
            gt_transform=default_style_gt_table,
        )
    )

## Parameter recovery


In [ ]:
RECOVERY_PARAMETER_SEED = 42
RECOVERY_N_EPISODES = None
POSTERIOR_SAMPLE_SEED = 42
NUM_POSTERIOR_SAMPLES = 2**9
MAX_SAMPLING_TIME_SECONDS = 60
POSTERIOR_DEVICE = "cpu"

In [ ]:
def resolve_existing_path(raw_path):
    path = Path(raw_path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.append(project_root / path)
        candidates.append(artifact_dir / path)
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return path.resolve()


def n_agents_values(raw_n_agents):
    if isinstance(raw_n_agents, dict):
        if raw_n_agents.get("kind") != "sampled_discrete_range":
            raise ValueError(f"Unsupported n_agents spec: {raw_n_agents}")
        return list(
            range(
                int(raw_n_agents["min"]),
                int(raw_n_agents["max"]) + 1,
                int(raw_n_agents.get("step", 1)),
            )
        )
    if isinstance(raw_n_agents, list):
        return [int(value) for value in raw_n_agents]
    return [int(raw_n_agents)]


def build_recovery_config(source_config, *, seed, n_episodes=None):
    config = copy.deepcopy(source_config)
    rng = np.random.default_rng(seed)

    population = config["population"]
    chosen_n_agents = int(rng.choice(n_agents_values(population["n_agents"])))
    proportion_spec = population["policy_proportions"]
    chosen_proportion = float(rng.choice(proportion_spec["values"]))

    config["sampling"]["log2_budget"] = 0
    config["sampling"]["seed"] = int(seed)
    population["n_agents"] = chosen_n_agents
    proportion_spec["values"] = [chosen_proportion]
    config["simulator"]["seed"] = int(source_config["simulator"]["seed"]) + 10000
    if n_episodes is not None:
        config["simulator"]["n_episodes"] = int(n_episodes)

    choices = {
        "recovery_parameter_seed": seed,
        "n_agents": chosen_n_agents,
        "policy_proportion": chosen_proportion,
        "simulator_seed": config["simulator"]["seed"],
        "n_episodes": config["simulator"]["n_episodes"],
    }
    return config, choices


def extract_observation(outputs, artifact):
    representation = artifact.metadata["dataset"].get("x_representation", "final")
    arrays = []
    for name in artifact.observation_order:
        if name not in outputs:
            raise KeyError(f"Recovery simulation did not produce {name!r}")
        array = np.asarray(outputs[name], dtype=np.float32)
        if representation == "final":
            arrays.append(float(array.reshape(-1)[-1]))
        elif representation == "timeseries":
            arrays.append(array)
        else:
            raise ValueError(f"Unsupported x representation: {representation}")

    if representation == "final":
        return np.asarray(arrays, dtype=np.float32)
    return np.stack(arrays, axis=-1).astype(np.float32, copy=False)


def theta_from_configuration(prepared, configuration, theta_order):
    values_by_name = {
        name: float(configuration[index])
        for name, index in prepared.sweep.column_index_by_name.items()
    }
    missing = [name for name in theta_order if name not in values_by_name]
    if missing:
        raise KeyError(f"Theta names missing from recovery config: {missing}")
    return np.asarray([values_by_name[name] for name in theta_order], dtype=np.float32)


source_config_path = resolve_existing_path(dataset_metadata["config_path"])
source_config = json.loads(source_config_path.read_text(encoding="utf-8"))
recovery_config, recovery_choices = build_recovery_config(
    source_config,
    seed=RECOVERY_PARAMETER_SEED,
    n_episodes=RECOVERY_N_EPISODES,
)
display(recovery_choices)

In [ ]:
prepared_recovery = prepare_policy_run(recovery_config)
recovery_outputs = prepared_recovery.run(0)
recovery_configuration = prepared_recovery.sweep.configuration_at(0)

true_theta = theta_from_configuration(
    prepared_recovery,
    recovery_configuration,
    artifact.theta_order,
)
recovery_x = extract_observation(recovery_outputs, artifact)
expected_x_shape = tuple(dataset_metadata["x_event_shape"])
if tuple(recovery_x.shape) != expected_x_shape:
    raise ValueError(
        f"recovery_x shape {tuple(recovery_x.shape)} does not match "
        f"expected shape {expected_x_shape}"
    )

display(
        make_gt_tabs(
            [
                {
                    "tab": "True theta",
                    "title": "True theta",
                    "value": [
                        {"theta": name, "true_value": float(value)}
                        for name, value in zip(artifact.theta_order, true_theta, strict=True)
                    ],
                },
                {
                    "tab": "Generated observation",
                    "title": "Generated observation",
                    "value": [
                        {"observable": name, "value": float(value)}
                        for name, value in zip(
                            artifact.observation_order,
                            np.ravel(recovery_x),
                            strict=True,
                        )
                    ] if recovery_x.ndim == 1 else [
                        {"x_index": index, "value": float(value)}
                        for index, value in enumerate(np.ravel(recovery_x))
                    ],
                }
            ],
            tab_key="tab",
            title_key="title",
            gt_transform=default_style_gt_table,
        )
    )

In [ ]:
torch.manual_seed(POSTERIOR_SAMPLE_SEED)
np.random.seed(POSTERIOR_SAMPLE_SEED)
x_tensor = torch.as_tensor(
    recovery_x,
    dtype=torch.float32,
    device=torch.device(POSTERIOR_DEVICE),
)
posterior_samples_tensor = artifact.posterior.sample(
    (NUM_POSTERIOR_SAMPLES,),
    x=x_tensor,
    max_sampling_time=MAX_SAMPLING_TIME_SECONDS,
    return_partial_on_timeout=True,
    show_progress_bars=True,
)
posterior_samples = np.asarray(
    posterior_samples_tensor.detach().cpu().numpy(),
    dtype=np.float32,
)
if posterior_samples.ndim == 1:
    posterior_samples = posterior_samples.reshape(-1, len(artifact.theta_order))
if posterior_samples.shape[0] == 0:
    raise RuntimeError(
        "Posterior sampling returned no samples before max_sampling_time. "
        "Try a longer MAX_SAMPLING_TIME_SECONDS or fewer recovery constraints."
    )

{
    "requested_samples": NUM_POSTERIOR_SAMPLES,
    "returned_samples": int(posterior_samples.shape[0]),
    "theta_dim": int(posterior_samples.shape[1]),
}

In [ ]:
posterior_mean = np.mean(posterior_samples, axis=0)
posterior_median = np.median(posterior_samples, axis=0)
posterior_q05 = np.quantile(posterior_samples, 0.05, axis=0)
posterior_q95 = np.quantile(posterior_samples, 0.95, axis=0)

recovery_summary = []
for index, name in enumerate(artifact.theta_order):
    recovery_summary.append(
        {
            "theta": name,
            "true": float(true_theta[index]),
            "mean": float(posterior_mean[index]),
            "median": float(posterior_median[index]),
            "q05": float(posterior_q05[index]),
            "q95": float(posterior_q95[index]),
            "abs_error_mean": float(abs(posterior_mean[index] - true_theta[index])),
            "covered_by_90_interval": bool(
                posterior_q05[index] <= true_theta[index] <= posterior_q95[index]
            ),
        }
    )

display(
    make_gt_tabs(
        [
            {
                "tab": "",
                "title": "Recovery summary",
                "value": recovery_summary,
            }
        ],
        tab_key="tab",
        title_key="title",
        gt_transform=default_style_gt_table,
    )
)

In [ ]:
n_theta = len(artifact.theta_order)
n_cols = min(3, n_theta)
n_rows = math.ceil(n_theta / n_cols)
figure, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = np.asarray(axes).reshape(-1)

for index, name in enumerate(artifact.theta_order):
    axis = axes[index]
    axis.hist(posterior_samples[:, index], bins=40, density=True, alpha=0.75)
    axis.axvline(true_theta[index], color="black", linewidth=2, label="true")
    axis.axvline(posterior_mean[index], color="tab:red", linestyle="--", label="mean")
    axis.set_title(name)
    axis.grid(alpha=0.2)
for axis in axes[n_theta:]:
    axis.axis("off")
axes[0].legend()
figure.tight_layout()
plt.show()

In [ ]:
if n_theta < 2:
    display(Markdown("Pair plot requires at least two theta dimensions."))
else:
    figure, axes = plt.subplots(
        n_theta,
        n_theta,
        figsize=(2.4 * n_theta, 2.4 * n_theta),
        squeeze=False,
    )
    for row in range(n_theta):
        for col in range(n_theta):
            axis = axes[row, col]
            if row == col:
                axis.hist(posterior_samples[:, col], bins=30, alpha=0.75)
                axis.axvline(true_theta[col], color="black", linewidth=1.5)
            else:
                axis.scatter(
                    posterior_samples[:, col],
                    posterior_samples[:, row],
                    s=6,
                    alpha=0.25,
                    linewidths=0,
                )
                axis.axvline(true_theta[col], color="black", linewidth=1.0)
                axis.axhline(true_theta[row], color="black", linewidth=1.0)
            if row == n_theta - 1:
                axis.set_xlabel(artifact.theta_order[col], fontsize=8)
            else:
                axis.set_xticklabels([])
            if col == 0:
                axis.set_ylabel(artifact.theta_order[row], fontsize=8)
            else:
                axis.set_yticklabels([])
            axis.grid(alpha=0.15)
    figure.tight_layout()
    plt.show()